# Offline Learning Module for Hybrid ALNS: Repair-Ranking Model Training

---

## I. Objective and Scope

This notebook specifies and executes the **offline supervised-learning pipeline** used to train the repair model consumed by the hybrid ALNS solver.

Given ALNS states and candidate insertions, the learning objective is to estimate a score function
$ f_	heta(\phi(x)) $ that ranks feasible repair actions by expected improvement quality. The trained artifact is serialized as `repair_model.pkl`.

This notebook covers only the offline stage:
- generation of labeled state-action examples,
- training/validation of the ranking surrogate,
- model persistence for downstream search-time inference.


## II. Data-Generation Protocol

Let $x_t$ denote a partially destroyed solution state at ALNS iteration $t$, and let $\mathcal{A}(x_t)$ be the set of feasible repair actions. Data generation samples tuples
$(x_t, a, y)$, where $a \in \mathcal{A}(x_t)$ and label $y$ encodes relative quality/acceptability under the training objective used by `collect_alns_states.py`.

Important protocol choices for statistical robustness:
- instance-size diversity (`--n-min`, `--n-max`),
- sufficient trajectory coverage (`--instances`),
- controlled negative sampling (`--max-negatives`),
- reproducibility (`--seed`).


## III. Recommended Training Configuration (Primary Recipe)

The following configuration is the default high-quality recipe for tabular ALNS features:
- model family: gradient-boosted trees (`--model xgb`),
- boosting depth/learning tradeoff: `--max-depth 6`, `--learning-rate 0.05`,
- ensemble capacity: `--n-estimators 400`,
- stochastic regularization: `--subsample 0.9`, `--colsample-bytree 0.8`.

Rationale: this configuration provides a strong bias-variance compromise for heterogeneous combinatorial-state descriptors while preserving ranking stability across unseen instances.


In [ ]:
!python collect_alns_states.py \
    --instances-dir ../instances \
    --output states_dataset.csv \
    --n-instances 500 \
    --iters 300 \
    --seed 42

In [ ]:
!python train_repair_model.py \
    --data states_dataset.csv \
    --out repair_model.pkl \
    --model xgb \
    --n-estimators 400 \
    --max-depth 6 \
    --learning-rate 0.05 \
    --subsample 0.9 \
    --colsample-bytree 0.8 \
    --seed 42

## IV. Secondary Baseline Configuration

A lighter baseline is provided for quick sanity checks and ablation comparisons. Its purpose is **not** to replace the primary recipe, but to verify that the feature pipeline, labels, and serialization flow are functioning correctly under reduced training cost.


In [ ]:
!python train_repair_model.py \
    --data states_dataset.csv \
    --out repair_model_quick.pkl \
    --model rf \
    --n-estimators 200 \
    --max-depth 12 \
    --seed 42

## V. Output Contract and Hand-Off to Runtime

Successful execution must produce `repair_model.pkl` in this directory (or the configured output path). The runtime notebooks (`benchmarking.ipynb` and `parameter_tuning.ipynb`) assume this artifact is present and loadable.

Before proceeding, verify:
1. training finished without exceptions,
2. validation/test metrics are reported,
3. output model path exists and matches runtime command arguments.
